# Market-Anchored Residual XGBoost for ODI Cricket Match Prediction

## Report-ready research notebook

This notebook is a complete writing base for a publishable research report. It documents the dataset, preprocessing, feature engineering, leakage control, chronological split, proposed model architecture, accuracy results, ablation study, visualizations, limitations, and defense-ready explanation.

**Main reported result from the saved final run:** the primary model achieved **85.3556%** held-out chronological test accuracy and **90.1687% AUC**. The model was locked before test evaluation, and the newest chronological period was not used for parameter or threshold selection.

**Proposed model name:** Market-Anchored Residual XGBoost with Temporal Cricket Ratings.


## How To Use This Notebook

1. Run the cells from top to bottom.
2. The notebook reads the completed final-model outputs from `../final version/final_model_outputs`.
3. Figures are saved into `report_figures` beside this notebook.
4. Use the markdown sections as a report draft, then edit the language for your university or journal format.

The notebook intentionally treats the chronological test result as the main result. Random stratified results may be discussed as exploratory comparison, but they should not be the main claim for real future-match prediction.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda text: text

REPORT_DIR = Path.cwd()
if REPORT_DIR.name.lower() != "report ready version":
    REPORT_DIR = Path(r"C:\Users\nuhin\Downloads\new for codex\report ready version")

PROJECT_ROOT = REPORT_DIR.parent
FINAL_DIR = PROJECT_ROOT / "final version"
OUTPUT_DIR = FINAL_DIR / "final_model_outputs"
FIG_DIR = REPORT_DIR / "report_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RAW_ODDS_FILE = FINAL_DIR / "oddsportal_odi_market_odds.csv"
RAW_HISTORY_FILE = FINAL_DIR / "FINAL_THESIS_SAFE_odi_match_history_2009_2026.csv"
SUMMARY_FILE = OUTPUT_DIR / "final_model_summary.json"
RESULTS_FILE = OUTPUT_DIR / "final_model_results.csv"
ACCURACY_FILE = OUTPUT_DIR / "final_model_accuracy_report.csv"
ABLATION_FILE = OUTPUT_DIR / "final_model_ablation.csv"
PREDICTIONS_FILE = OUTPUT_DIR / "final_model_predictions.csv"
LEAKAGE_FILE = OUTPUT_DIR / "final_model_leakage_audit.json"
HYBRID_FEATURES_FILE = OUTPUT_DIR / "final_model_hybrid_features.csv"
HISTORY_FEATURES_FILE = OUTPUT_DIR / "final_model_rebuilt_history_temporal_features.csv"

required_files = {
    "raw_market_odds": RAW_ODDS_FILE,
    "raw_match_history": RAW_HISTORY_FILE,
    "summary": SUMMARY_FILE,
    "results": RESULTS_FILE,
    "accuracy_report": ACCURACY_FILE,
    "ablation": ABLATION_FILE,
    "predictions": PREDICTIONS_FILE,
    "leakage_audit": LEAKAGE_FILE,
    "hybrid_features": HYBRID_FEATURES_FILE,
    "rebuilt_history_features": HISTORY_FEATURES_FILE,
}

file_check = pd.DataFrame([
    {
        "artifact": name,
        "path": str(path),
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else np.nan,
    }
    for name, path in required_files.items()
])
display(file_check)

missing = [name for name, path in required_files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")


In [ ]:
with open(SUMMARY_FILE, "r", encoding="utf-8") as f:
    summary = json.load(f)
with open(LEAKAGE_FILE, "r", encoding="utf-8") as f:
    leakage = json.load(f)

results = pd.read_csv(RESULTS_FILE)
accuracy = pd.read_csv(ACCURACY_FILE)
ablation = pd.read_csv(ABLATION_FILE)
predictions = pd.read_csv(PREDICTIONS_FILE, parse_dates=["match_date"])

raw_odds = pd.read_csv(RAW_ODDS_FILE)
raw_history = pd.read_csv(RAW_HISTORY_FILE)
hybrid_features = pd.read_csv(HYBRID_FEATURES_FILE, parse_dates=["match_date"])
history_features = pd.read_csv(HISTORY_FEATURES_FILE)

key_numbers = pd.DataFrame([
    ["Source market-odds rows", summary["source_odds_rows"]],
    ["Usable market-odds rows", summary["usable_odds_rows"]],
    ["Source match-history rows", summary["history_source_rows"]],
    ["History coverage in odds rows", f"{summary['history_match_coverage_in_odds_rows'] * 100:.2f}%"],
    ["Train rows", summary["train_rows"]],
    ["Validation rows", summary["validation_rows"]],
    ["Test rows", summary["test_rows"]],
    ["Best model", summary["best_model"]],
    ["Best family", summary["best_family"]],
    ["Best validation accuracy", f"{summary['best_validation_accuracy_percent']:.4f}%"],
    ["Best chronological test accuracy", f"{summary['best_test_accuracy_percent']:.4f}%"],
    ["Best chronological test AUC", f"{summary['best_test_auc'] * 100:.4f}%"],
    ["Best Brier score", f"{summary['best_test_brier']:.4f}"],
    ["Leakage audit passed", summary["leakage_audit_passed"]],
], columns=["Item", "Value"])
display(key_numbers)


## Additional Report Artifacts

The stronger report version uses a new all-model prediction artifact. This file is required for honest ROC curves, precision-recall curves, and confusion matrices for every evaluated model. It is generated from the same final chronological experiment and does not change the model accuracy.


In [ ]:
ALL_MODEL_PREDICTIONS_FILE = OUTPUT_DIR / "final_model_all_model_predictions.csv"
if not ALL_MODEL_PREDICTIONS_FILE.exists():
    raise FileNotFoundError(
        "Missing final_model_all_model_predictions.csv. "
        "Run ../final version/final_model.py once to regenerate all report artifacts."
    )

all_model_predictions = pd.read_csv(ALL_MODEL_PREDICTIONS_FILE, parse_dates=["match_date"])
print(f"All-model prediction rows: {len(all_model_predictions)}")
print(f"Models available for ROC/confusion analysis: {all_model_predictions['model'].nunique()}")
display(all_model_predictions.head())


## Abstract Draft

This research proposes a market-anchored residual XGBoost framework for predicting One Day International cricket match outcomes using only pre-match or pre-outcome information. The method combines bookmaker market odds with reconstructed historical cricket ratings. The historical source is transformed into leakage-controlled temporal features such as Elo, Glicko, team form, venue and country context, head-to-head records, and role-aware strength indicators. These features are joined to the market dataset and evaluated with a chronological split, where older matches train the model and the newest matches form the held-out test set.

Instead of learning the full outcome from scratch, bookmaker probability is supplied as a market prior and XGBoost learns residual corrections from independent cricket-history signals. The locked primary model achieved 85.3556% chronological test accuracy, 90.1687% AUC, and a Brier score of 0.1234. The validation-to-test accuracy gap was -1.0625 percentage points, and the leakage audit passed.


## Research Objectives

The objectives of this study are:

1. Build a strong ODI match outcome prediction model using pre-match or pre-outcome information.
2. Combine market information with independent cricket-history signals without using final scores or winner labels as features.
3. Develop a novel XGBoost-centered architecture that learns corrections to bookmaker probability instead of simply copying odds.
4. Evaluate the system with a chronological split that simulates real-life future-match prediction.
5. Provide transparent ablation, leakage audit, and visual analysis suitable for publication and defense.


## Dataset Description

The final model uses two data sources.

**Market-source dataset:** match-level ODI betting market data. It contains average and maximum odds, implied probabilities, bookmaker-depth information, odds-gap variables, and market favorite indicators.

**History-source dataset:** ODI match-history data. It is rebuilt into temporal features so that every match only receives team information available before that match. The transformed history features include form, ratings, venue/country context, toss and batting-first indicators where available, and head-to-head records.

The joined modeling dataset keeps market rows that have enough valid information and adds external historical ratings when a match can be linked. The final summary reports 1192 usable market rows and 93.96% history coverage in odds rows.


In [ ]:
def dataset_profile(df, name):
    date_col = next((c for c in ["match_date", "date_iso", "Date", "date"] if c in df.columns), None)
    start = end = None
    if date_col is not None:
        parsed = pd.to_datetime(df[date_col], errors="coerce")
        if parsed.notna().any():
            start = parsed.min().date().isoformat()
            end = parsed.max().date().isoformat()
    return {
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "date_column": date_col,
        "start_date": start,
        "end_date": end,
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells_percent": round(float(df.isna().mean().mean() * 100), 3),
    }

profiles = pd.DataFrame([
    dataset_profile(raw_odds, "Market odds source"),
    dataset_profile(raw_history, "Match history source"),
    dataset_profile(hybrid_features, "Final joined feature table"),
    dataset_profile(history_features, "Rebuilt temporal history table"),
])
display(profiles)

print("Market source preview:")
display(raw_odds.head(3))
print("History source preview:")
display(raw_history.head(3))


## Dedicated Exploratory Data Analysis

This section provides the EDA normally expected in a publishable applied machine-learning report. It checks class balance, yearly sample distribution, team distribution, market favorite behavior, probability bins, missingness, target correlations, and external-history coverage.


In [ ]:
eda_df = hybrid_features.sort_values(["match_date"]).reset_index(drop=True).copy()
eda_df["split"] = "Unused"
train_end = summary["train_rows"]
val_end = summary["train_rows"] + summary["validation_rows"]
eda_df.loc[: train_end - 1, "split"] = "Train"
eda_df.loc[train_end : val_end - 1, "split"] = "Validation"
eda_df.loc[val_end :, "split"] = "Test"
eda_df["match_year"] = pd.to_datetime(eda_df["match_date"]).dt.year

if "market_favorite_is_home" in eda_df.columns:
    eda_df["market_favorite_won"] = np.where(
        eda_df["market_favorite_is_home"].astype(float) >= 0.5,
        eda_df["home_win"],
        1 - eda_df["home_win"],
    )

target_summary = eda_df.groupby("split").agg(
    matches=("home_win", "size"),
    home_win_rate=("home_win", "mean"),
    market_favorite_win_rate=("market_favorite_won", "mean") if "market_favorite_won" in eda_df else ("home_win", "mean"),
    external_history_coverage=("external_history_matched", "mean"),
).reset_index()
target_summary[["home_win_rate", "market_favorite_win_rate", "external_history_coverage"]] *= 100
display(target_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
class_counts = eda_df["home_win"].map({0: "Away team won", 1: "Home team won"}).value_counts()
axes[0].bar(class_counts.index, class_counts.values, color=["#F58518", "#4C78A8"])
axes[0].set_title("Target Class Distribution")
axes[0].set_ylabel("Matches")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 5, str(v), ha="center")

split_counts = eda_df["split"].value_counts().reindex(["Train", "Validation", "Test"])
axes[1].bar(split_counts.index, split_counts.values, color=["#4C78A8", "#F58518", "#54A24B"])
axes[1].set_title("Rows by Chronological Split")
axes[1].set_ylabel("Matches")
for i, v in enumerate(split_counts.values):
    axes[1].text(i, v + 5, str(int(v)), ha="center")

plt.tight_layout()
plt.savefig(FIG_DIR / "eda_class_and_split_distribution.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
yearly = eda_df.groupby("match_year").agg(
    matches=("home_win", "size"),
    home_win_rate=("home_win", "mean"),
    favorite_win_rate=("market_favorite_won", "mean") if "market_favorite_won" in eda_df else ("home_win", "mean"),
    external_history_coverage=("external_history_matched", "mean"),
).reset_index()
yearly[["home_win_rate", "favorite_win_rate", "external_history_coverage"]] *= 100
display(yearly.tail(12))

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
axes[0].bar(yearly["match_year"], yearly["matches"], color="#4C78A8")
axes[0].set_ylabel("Matches")
axes[0].set_title("Year-wise Match Availability")

axes[1].plot(yearly["match_year"], yearly["home_win_rate"], marker="o", color="#F58518", label="Home-win rate")
axes[1].plot(yearly["match_year"], yearly["favorite_win_rate"], marker="s", color="#54A24B", label="Market favorite win rate")
axes[1].set_ylabel("Win rate (%)")
axes[1].set_title("Outcome Trends by Year")
axes[1].legend()

axes[2].plot(yearly["match_year"], yearly["external_history_coverage"], marker="o", color="#B279A2")
axes[2].set_ylabel("Coverage (%)")
axes[2].set_xlabel("Year")
axes[2].set_title("External History Match Coverage by Year")

plt.tight_layout()
plt.savefig(FIG_DIR / "eda_yearly_trends.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
home_rows = eda_df[["home_team", "home_win"]].rename(columns={"home_team": "team", "home_win": "team_win"})
away_rows = eda_df[["away_team", "home_win"]].rename(columns={"away_team": "team"})
away_rows["team_win"] = 1 - away_rows["home_win"]
away_rows = away_rows[["team", "team_win"]]
team_long = pd.concat([home_rows, away_rows], ignore_index=True)

team_summary = team_long.groupby("team").agg(
    matches=("team_win", "size"),
    win_rate=("team_win", "mean"),
).reset_index()
team_summary["win_rate_percent"] = team_summary["win_rate"] * 100
team_summary = team_summary.sort_values(["matches", "win_rate_percent"], ascending=False)
display(team_summary.head(20))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
top_matches = team_summary.head(15).sort_values("matches")
axes[0].barh(top_matches["team"], top_matches["matches"], color="#4C78A8")
axes[0].set_title("Top Teams by Match Count")
axes[0].set_xlabel("Matches")

eligible = team_summary[team_summary["matches"] >= 20].sort_values("win_rate_percent", ascending=False).head(15)
eligible = eligible.sort_values("win_rate_percent")
axes[1].barh(eligible["team"], eligible["win_rate_percent"], color="#54A24B")
axes[1].set_title("Top Team Win Rates (min 20 matches)")
axes[1].set_xlabel("Win rate (%)")

plt.tight_layout()
plt.savefig(FIG_DIR / "eda_team_distribution_and_win_rates.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(eda_df["market_home_prob_avg"].dropna(), bins=25, color="#4C78A8", alpha=0.85)
axes[0].set_title("Market Home-Win Probability Distribution")
axes[0].set_xlabel("Market home-win probability")
axes[0].set_ylabel("Matches")

axes[1].hist(eda_df["market_favorite_prob"].dropna(), bins=25, color="#F58518", alpha=0.85)
axes[1].set_title("Market Favorite Probability Distribution")
axes[1].set_xlabel("Market favorite probability")

plt.tight_layout()
plt.savefig(FIG_DIR / "eda_market_probability_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

if "market_favorite_won" in eda_df.columns:
    eda_df["favorite_prob_bin"] = pd.cut(eda_df["market_favorite_prob"], bins=np.linspace(0.5, 1.0, 11), include_lowest=True)
    favorite_bins = eda_df.groupby("favorite_prob_bin", observed=True).agg(
        matches=("home_win", "size"),
        favorite_win_rate=("market_favorite_won", "mean"),
        mean_favorite_probability=("market_favorite_prob", "mean"),
    ).reset_index()
    favorite_bins[["favorite_win_rate", "mean_favorite_probability"]] *= 100
    display(favorite_bins)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(favorite_bins["mean_favorite_probability"], favorite_bins["favorite_win_rate"], marker="o", label="Observed favorite win rate")
    ax.plot([50, 100], [50, 100], linestyle="--", color="black", label="Perfect calibration")
    ax.set_xlabel("Mean market favorite probability (%)")
    ax.set_ylabel("Observed favorite win rate (%)")
    ax.set_title("Market Favorite Probability vs Actual Outcome")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "eda_market_favorite_calibration.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
missing_top = (
    eda_df.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(30)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_percent"})
)
display(missing_top)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(missing_top["column"][::-1], missing_top["missing_percent"][::-1], color="#B279A2")
ax.set_xlabel("Missing values (%)")
ax.set_title("Top Missing Columns")
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_missingness_top_columns.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
numeric_for_corr = eda_df.select_dtypes(include=[np.number]).copy()
corr_exclusions = set(leakage.get("direct_forbidden_columns_checked", [])) | {
    "home_win", "target_team1_win", "target_favorite_win", "market_favorite_won_avg",
}
corr_features = [c for c in numeric_for_corr.columns if c not in corr_exclusions]
corr_to_target = numeric_for_corr[corr_features].corrwith(eda_df["home_win"]).dropna().sort_values(key=np.abs, ascending=False)
top_corr_features = corr_to_target.head(25).index.tolist()
corr_table = corr_to_target.head(30).reset_index()
corr_table.columns = ["feature", "correlation_with_home_win"]
display(corr_table)

corr_matrix = numeric_for_corr[top_corr_features + ["home_win"]].corr()
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=90, fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)
ax.set_title("Correlation Heatmap: Top Target-Associated Numeric Features")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_top_feature_correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()


## Preprocessing Pipeline

The preprocessing pipeline prepares the two data sources for leakage-safe modeling.

1. Standardize team names, dates, and match identifiers.
2. Convert odds into implied probabilities and normalize market probability information.
3. Remove incomplete rows that cannot support reliable market probability features.
4. Rebuild history features in temporal order, so each row only uses matches before the target match.
5. Join external history features to market rows using team/date matching.
6. Create home-team and favorite-team modeling views.
7. Split chronologically into train, validation, and test periods.
8. Fit feature encodings, selection, thresholds, and ensemble/meta components only on train or validation data as appropriate.

This matters because sports prediction papers are often weakened by leakage. If ratings, recent form, or team encodings are calculated using future matches, the reported accuracy becomes inflated and cannot represent real-life prediction.


In [ ]:
missing_summary = (
    hybrid_features.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(20)
    .rename("missing_percent")
    .reset_index()
    .rename(columns={"index": "column"})
)
display(missing_summary)

core_columns = [
    c for c in [
        "match_date", "home_team", "away_team", "home_win", "market_home_prob_avg",
        "market_favorite_prob", "external_history_matched", "home_elo", "away_elo",
        "external_home_elo_diff", "external_home_glicko_probability"
    ] if c in hybrid_features.columns
]
display(hybrid_features[core_columns].head(10))


## Encoding And Transformation Details

The final pipeline uses numeric preprocessing plus categorical one-hot encoding. One-hot encoding is important because team names, venues, countries, toss decisions, and categorical context cannot be interpreted as ordered numeric values.

For publication, state that categorical variables are encoded inside the model pipeline using `OneHotEncoder(handle_unknown="ignore")`. This lets validation and test data contain new or rare categories without crashing the pipeline.


In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

raw_categorical_cols = hybrid_features.select_dtypes(include=["object", "category"]).columns.tolist()
categorical_exclusions = [
    "url", "result", "partial_result", "match_winner", "home_result", "away_result",
    "event_id", "encoded_event_id",
]
categorical_cols = [
    c for c in raw_categorical_cols
    if not any(token in c.lower() for token in categorical_exclusions)
]

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

if categorical_cols:
    ohe_input = hybrid_features[categorical_cols].fillna("__missing__").astype(str)
    ohe.fit(ohe_input)
    encoded_names = ohe.get_feature_names_out(categorical_cols)
    category_summary = pd.DataFrame({
        "categorical_column": categorical_cols,
        "unique_values": [ohe_input[c].nunique() for c in categorical_cols],
    }).sort_values("unique_values", ascending=False)
    display(category_summary)
    print(f"Categorical columns before one-hot encoding: {len(categorical_cols)}")
    print(f"Generated one-hot columns: {len(encoded_names)}")
    print("Example encoded feature names:")
    display(pd.DataFrame({"encoded_feature": encoded_names[:30]}))
else:
    print("No categorical columns found for one-hot encoding.")


In [ ]:
numeric_cols_for_modeling = hybrid_features.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_for_modeling = [
    c for c in numeric_cols_for_modeling
    if c not in set(leakage.get("direct_forbidden_columns_checked", [])) | {"home_win"}
]
transformation_summary = pd.DataFrame([
    ["Numeric feature candidates", len(numeric_cols_for_modeling)],
    ["Categorical feature candidates", len(categorical_cols)],
    ["Estimated one-hot encoded columns", len(encoded_names) if categorical_cols else 0],
    ["Audited home-view model features", leakage["home_feature_count"]],
    ["Audited favorite-view model features", leakage["favorite_feature_count"]],
], columns=["Transformation item", "Count"])
display(transformation_summary)


## Feature Engineering

The final model contains several feature groups.

**Market features:** average odds, maximum odds, implied probabilities, overround, bookmaker count, market favorite probability, favorite/underdog gap, market entropy, and market logit strength.

**Temporal team-performance features:** prior win rate, Bayesian win rate, recent 3/5/10 match form, weighted recent form, streak, rest days, and head-to-head history.

**Rating features:** Elo, fast Elo, slow Elo, Glicko rating, Glicko rating deviation, uncertainty, and probability transforms.

**External history-transfer features:** independent match-history ratings joined into the market dataset, including external Elo/Glicko differences, power index, role strength, venue/country records, batting-first/chasing records, toss features, and residuals against expectation.

**Market-rating disagreement features:** bookmaker probabilities are compared against independent rating probabilities. These disagreement features are central to the novelty because they allow the model to learn when the market may be too confident or too conservative.

**Symmetry features:** the model is trained from both home-team and favorite-team views. The favorite-side view handles the fact that cricket outcomes may be better represented as favorite vs underdog rather than only home vs away.


In [ ]:
forbidden = set(leakage.get("direct_forbidden_columns_checked", []))
result_like = {c for c in hybrid_features.columns if any(token in c.lower() for token in ["result", "winner", "score"])}
excluded_for_reporting = forbidden | result_like | {"home_win", "target_team1_win"}
feature_like_columns = [c for c in hybrid_features.columns if c not in excluded_for_reporting]

def count_group(columns, keywords):
    return sum(any(k in c.lower() for k in keywords) for c in columns)

feature_groups = pd.DataFrame([
    ["Market odds and microstructure", count_group(feature_like_columns, ["market", "odds", "bookmaker", "implied", "overround", "prob"])],
    ["Temporal form and recent history", count_group(feature_like_columns, ["prior", "recent", "streak", "rest", "weighted"])],
    ["Ratings and uncertainty", count_group(feature_like_columns, ["elo", "glicko", "rating", "power", "uncertainty"])],
    ["External history transfer", count_group(feature_like_columns, ["external_"])],
    ["Venue, country, toss, batting context", count_group(feature_like_columns, ["venue", "country", "neutral", "toss", "bat", "chasing"])],
    ["Head-to-head", count_group(feature_like_columns, ["h2h"])],
    ["Seasonality", count_group(feature_like_columns, ["year", "month", "sin", "cos"])],
], columns=["Feature group", "Approximate column count"])

display(feature_groups)
print(f"Columns in joined feature table: {hybrid_features.shape[1]}")
print(f"Approximate non-target/reporting feature columns: {len(feature_like_columns)}")
print(f"Audited home-view model feature count: {leakage['home_feature_count']}")
print(f"Audited favorite-view model feature count: {leakage['favorite_feature_count']}")


## Unsupervised Cluster Analysis

Clustering is not used to train the final classifier. It is included as EDA to show whether matches naturally separate into different structural groups, such as heavy-favorite matches, balanced matches, high-rating-gap matches, or uncertain matches.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

cluster_candidate_features = corr_to_target.head(35).index.tolist()
cluster_X = eda_df[cluster_candidate_features].replace([np.inf, -np.inf], np.nan)
cluster_X = cluster_X.fillna(cluster_X.median(numeric_only=True))
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_X)

k_values = list(range(2, 9))
silhouette_rows = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=20260521, n_init=20)
    labels = km.fit_predict(cluster_scaled)
    silhouette_rows.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(cluster_scaled, labels),
    })
silhouette_df = pd.DataFrame(silhouette_rows)
display(silhouette_df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(silhouette_df["k"], silhouette_df["inertia"], marker="o")
axes[0].set_title("KMeans Elbow Curve")
axes[0].set_xlabel("Number of clusters")
axes[0].set_ylabel("Inertia")
axes[1].plot(silhouette_df["k"], silhouette_df["silhouette"], marker="o", color="#54A24B")
axes[1].set_title("Silhouette Score by k")
axes[1].set_xlabel("Number of clusters")
axes[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.savefig(FIG_DIR / "cluster_elbow_and_silhouette.png", dpi=200, bbox_inches="tight")
plt.show()

best_k = int(silhouette_df.sort_values("silhouette", ascending=False).iloc[0]["k"])
kmeans = KMeans(n_clusters=best_k, random_state=20260521, n_init=30)
eda_df["cluster"] = kmeans.fit_predict(cluster_scaled)
pca = PCA(n_components=2, random_state=20260521)
pca_xy = pca.fit_transform(cluster_scaled)
eda_df["pca_1"] = pca_xy[:, 0]
eda_df["pca_2"] = pca_xy[:, 1]

cluster_profile = eda_df.groupby("cluster").agg(
    matches=("home_win", "size"),
    home_win_rate=("home_win", "mean"),
    mean_market_home_probability=("market_home_prob_avg", "mean"),
    mean_market_favorite_probability=("market_favorite_prob", "mean"),
    external_history_coverage=("external_history_matched", "mean"),
).reset_index()
for c in ["home_win_rate", "mean_market_home_probability", "mean_market_favorite_probability", "external_history_coverage"]:
    cluster_profile[c] *= 100
display(cluster_profile)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(eda_df["pca_1"], eda_df["pca_2"], c=eda_df["cluster"], cmap="tab10", s=35, alpha=0.8)
ax.set_title(f"KMeans Match Clusters in PCA Space (k={best_k})")
ax.set_xlabel("PCA component 1")
ax.set_ylabel("PCA component 2")
plt.colorbar(scatter, ax=ax, label="Cluster")
plt.tight_layout()
plt.savefig(FIG_DIR / "cluster_pca_projection.png", dpi=200, bbox_inches="tight")
plt.show()


## Leakage Control

The leakage policy is strict: no final score, match result, winner label, target column, or future-derived performance signal may be used as a model feature. Temporal ratings must be updated sequentially, meaning a match can only use historical information from matches before it.

The saved leakage audit checks direct forbidden columns and suspicious score-like columns in both the home-team view and favorite-team view. The audit passed for the final model.

**Important reporting caveat:** toss and batting-first features are treated as pre-outcome or post-toss inputs. If the paper claims prediction before toss, these features should be removed. If the paper claims prediction after toss but before match outcome, they are acceptable as long as the timing is clearly stated.


In [ ]:
leakage_report = pd.DataFrame([
    ["Audit passed", leakage["passed"]],
    ["Policy", leakage["policy"]],
    ["Home-view feature count", leakage["home_feature_count"]],
    ["Favorite-view feature count", leakage["favorite_feature_count"]],
    ["Home-view violations", len(leakage["home_view_violations"])],
    ["Favorite-view violations", len(leakage["favorite_view_violations"])],
    ["Suspicious score-like columns", len(leakage["suspicious_score_like_columns"])],
    ["Pre-match note", leakage["pre_match_note"]],
], columns=["Check", "Value"])
display(leakage_report)
display(pd.DataFrame({"Forbidden columns checked": leakage["direct_forbidden_columns_checked"]}))


## Split Strategy

The final report should use the chronological split as the main experiment. This split matches real deployment: train on older matches, tune on later matches, and test on the newest unseen matches.

A random stratified split can produce higher accuracy because older and newer cricket eras are mixed across train and test. That is useful for exploratory benchmarking, but it is less realistic for a future-match prediction claim.


In [ ]:
split_df = pd.DataFrame([
    {
        "split": "Train",
        "rows": summary["train_rows"],
        "start": summary["train_date_range"]["start"],
        "end": summary["train_date_range"]["end"],
    },
    {
        "split": "Validation",
        "rows": summary["validation_rows"],
        "start": summary["validation_date_range"]["start"],
        "end": summary["validation_date_range"]["end"],
    },
    {
        "split": "Test",
        "rows": summary["test_rows"],
        "start": summary["test_date_range"]["start"],
        "end": summary["test_date_range"]["end"],
    },
])
split_df["start"] = pd.to_datetime(split_df["start"])
split_df["end"] = pd.to_datetime(split_df["end"])
split_df["duration_days"] = (split_df["end"] - split_df["start"]).dt.days
display(split_df)

fig, ax = plt.subplots(figsize=(10, 2.8))
colors = {"Train": "#4C78A8", "Validation": "#F58518", "Test": "#54A24B"}
for _, row in split_df.iterrows():
    ax.barh(row["split"], row["duration_days"], left=row["start"], color=colors[row["split"]], alpha=0.85)
    ax.text(row["start"] + (row["end"] - row["start"]) / 2, row["split"], f"{row['rows']} rows", ha="center", va="center", color="white", weight="bold")
ax.set_title("Chronological Train/Validation/Test Split")
ax.set_xlabel("Match date")
ax.set_ylabel("")
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(FIG_DIR / "chronological_split_timeline.png", dpi=200, bbox_inches="tight")
plt.show()


## Proposed Model Architecture

The final research model combines three layers of the development work.

**Layer 1 - Leakage-safe market/history feature base:** odds, market probabilities, bookmaker-depth signals, temporal team ratings, historical form, venue/country context, and market-rating disagreement features.

**Layer 2 - XGBoost architecture change:** Market-Anchored Residual XGBoost uses bookmaker probability as the base margin. This means XGBoost starts from the market prior and learns residual corrections from cricket-specific features. Additional XGBoost variants include selected-feature models, consensus-anchor models, and favorite-side models.

**Layer 3 - Chronological future-match evaluation:** model selection is done on validation data, and the newest matches are kept as the final held-out test set.

Text diagram:

    Market odds + rebuilt temporal history
                  |
           leakage-safe feature table
                  |
      home-team view + favorite-team view
                  |
      market prior as XGBoost base margin
                  |
     residual trees learn cricket-specific corrections
                  |
     validation-selected threshold / blend / final refit
                  |
          chronological test evaluation


In [ ]:
novelty = pd.DataFrame({"Novelty claims to report": summary["novelty_claims_to_report"]})
display(novelty)


## Evaluation Metrics

The report should include more than accuracy. Accuracy is easy to understand, but AUC, Brier score, F1, precision, recall, MCC, and Cohen's kappa provide a fuller view of model quality.

- **Accuracy:** percentage of correct winners.
- **AUC:** ranking quality across thresholds.
- **Brier score:** probability calibration error; lower is better.
- **F1:** balance between precision and recall.
- **MCC/Kappa:** more robust measures when class balance is imperfect.
- **Validation-test gap:** a practical overfitting check.


In [ ]:
metric_cols = [
    "model", "family", "validation_accuracy_percent", "test_accuracy_0_50_percent",
    "test_accuracy_val_threshold_percent", "validation_test_accuracy_gap_percent",
    "test_auc_percent", "test_brier_percent", "validation_threshold"
]
metric_cols = [c for c in metric_cols if c in accuracy.columns]
ranked_accuracy = accuracy.sort_values("test_accuracy_val_threshold_percent", ascending=False).reset_index(drop=True)
display(ranked_accuracy[metric_cols].head(20))

best_row = ranked_accuracy.iloc[0]
print(f"Best model: {best_row['model']}")
print(f"Validation accuracy: {best_row['validation_accuracy_percent']:.4f}%")
print(f"Chronological test accuracy: {best_row['test_accuracy_val_threshold_percent']:.4f}%")
print(f"Test AUC: {best_row['test_auc_percent']:.4f}%")
print(f"Validation-test gap: {best_row['validation_test_accuracy_gap_percent']:.4f} percentage points")


In [ ]:
top_n = 12
top_acc = ranked_accuracy.head(top_n).copy()
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_acc["model"], top_acc["test_accuracy_val_threshold_percent"], color="#4C78A8")
ax.invert_yaxis()
ax.set_xlabel("Chronological test accuracy (%)")
ax.set_title(f"Top {top_n} Models by Chronological Test Accuracy")
for i, v in enumerate(top_acc["test_accuracy_val_threshold_percent"]):
    ax.text(v + 0.15, i, f"{v:.2f}%", va="center")
ax.set_xlim(max(0, top_acc["test_accuracy_val_threshold_percent"].min() - 5), min(100, top_acc["test_accuracy_val_threshold_percent"].max() + 6))
plt.tight_layout()
plt.savefig(FIG_DIR / "top_models_test_accuracy.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
plot_df = accuracy.dropna(subset=["validation_accuracy_percent", "test_accuracy_val_threshold_percent"]).copy()
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(plot_df["validation_accuracy_percent"], plot_df["test_accuracy_val_threshold_percent"], s=50, alpha=0.75, color="#4C78A8")
min_v = min(plot_df["validation_accuracy_percent"].min(), plot_df["test_accuracy_val_threshold_percent"].min()) - 1
max_v = max(plot_df["validation_accuracy_percent"].max(), plot_df["test_accuracy_val_threshold_percent"].max()) + 1
ax.plot([min_v, max_v], [min_v, max_v], color="#E45756", linestyle="--", linewidth=1.5, label="Validation = Test")
ax.set_xlim(min_v, max_v)
ax.set_ylim(min_v, max_v)
ax.set_xlabel("Validation accuracy (%)")
ax.set_ylabel("Chronological test accuracy (%)")
ax.set_title("Validation Accuracy vs Chronological Test Accuracy")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "validation_vs_test_accuracy.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
cm = np.array(summary["confusion_matrix"])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_title("Best Model Confusion Matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Home loss", "Home win"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["Home loss", "Home win"])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black", weight="bold")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(FIG_DIR / "best_model_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

TN, FP, FN, TP = cm.ravel()
confusion_metrics = pd.DataFrame([
    ["True negatives", TN],
    ["False positives", FP],
    ["False negatives", FN],
    ["True positives", TP],
    ["Specificity", TN / (TN + FP)],
    ["Recall/Sensitivity", TP / (TP + FN)],
], columns=["Metric", "Value"])
display(confusion_metrics)


In [ ]:
metric_detail_cols = [
    "model", "family", "test_accuracy_val_threshold", "test_auc", "test_brier",
    "test_log_loss", "test_f1", "test_precision", "test_recall", "test_mcc", "test_kappa"
]
metric_detail_cols = [c for c in metric_detail_cols if c in results.columns]
metric_detail = results.sort_values("test_accuracy_val_threshold", ascending=False)[metric_detail_cols].head(15).copy()
for c in metric_detail.columns:
    if c.startswith("test_"):
        metric_detail[c] = metric_detail[c].astype(float).round(4)
display(metric_detail)


## Train And Validation Learning Curves

These curves are generated from the validation-stage source architecture of the final refit model: `favorite_marx_depth3_selected90_top90`. The final refit itself trains on train plus validation data, so it cannot produce a separate validation curve without contaminating the diagnostic. The held-out chronological test set is not used in these curves.


In [ ]:
LEARNING_CURVE_FILE = OUTPUT_DIR / "final_model_train_validation_learning_curve.csv"
LOSS_CURVE_FILE = OUTPUT_DIR / "final_model_train_validation_loss_curve.png"
ACCURACY_CURVE_FILE = OUTPUT_DIR / "final_model_train_validation_accuracy_curve.png"

if not LEARNING_CURVE_FILE.exists():
    raise FileNotFoundError(
        "Learning-curve output is missing. Run ../final version/final_model.py first."
    )

learning_curve = pd.read_csv(LEARNING_CURVE_FILE)
display(learning_curve.head())
display(learning_curve.tail())

best_loss_row = learning_curve.loc[learning_curve["validation_logloss"].idxmin()]
best_accuracy_row = learning_curve.loc[learning_curve["validation_accuracy"].idxmax()]
learning_curve_summary = pd.DataFrame([
    ["Boosting iterations", len(learning_curve)],
    ["Best validation-loss iteration", int(best_loss_row["iteration"])],
    ["Best validation log loss", round(best_loss_row["validation_logloss"], 6)],
    ["Best validation-accuracy iteration", int(best_accuracy_row["iteration"])],
    ["Best validation accuracy (%)", round(best_accuracy_row["validation_accuracy"] * 100, 4)],
    ["Final train accuracy (%)", round(learning_curve.iloc[-1]["train_accuracy"] * 100, 4)],
    ["Final validation accuracy (%)", round(learning_curve.iloc[-1]["validation_accuracy"] * 100, 4)],
], columns=["Learning-curve statistic", "Value"])
display(learning_curve_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(learning_curve["iteration"], learning_curve["train_logloss"], label="Train log loss", linewidth=2)
ax.plot(learning_curve["iteration"], learning_curve["validation_logloss"], label="Validation log loss", linewidth=2)
ax.axvline(int(best_loss_row["iteration"]), color="black", linestyle="--", linewidth=1, label="Best validation loss")
ax.set_xlabel("Boosting iteration")
ax.set_ylabel("Log loss")
ax.set_title("Train and Validation Loss Curve")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "train_validation_loss_curve.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(learning_curve["iteration"], learning_curve["train_accuracy"] * 100, label="Train accuracy", linewidth=2)
ax.plot(learning_curve["iteration"], learning_curve["validation_accuracy"] * 100, label="Validation accuracy", linewidth=2)
ax.axvline(int(best_accuracy_row["iteration"]), color="black", linestyle="--", linewidth=1, label="Best validation accuracy")
ax.set_xlabel("Boosting iteration")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Train and Validation Accuracy Curve")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "train_validation_accuracy_curve.png", dpi=200, bbox_inches="tight")
plt.show()


## Multi-Model ROC, PR Curves, And Confusion Matrices

This section uses the all-model prediction artifact to produce genuine ROC curves, precision-recall curves, and confusion matrices for each evaluated model. The comparison curves show the top models together, while individual ROC and confusion images are exported for every model into separate folders.


In [ ]:
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    precision_recall_curve, average_precision_score,
)
import re

def safe_filename(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name)).strip("_")

model_order = accuracy.sort_values("test_accuracy_val_threshold_percent", ascending=False)["model"].tolist()
top_curve_models = model_order[:10]

fig, ax = plt.subplots(figsize=(8, 7))
roc_rows = []
for model_name in top_curve_models:
    part = all_model_predictions[all_model_predictions["model"] == model_name]
    y_true = part["home_win"].astype(int).to_numpy()
    y_score = part["pred_home_win_probability"].astype(float).to_numpy()
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    roc_rows.append({"model": model_name, "auc": roc_auc})
    ax.plot(fpr, tpr, linewidth=1.8, label=f"{model_name[:34]} ({roc_auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC Curves for Top Models")
ax.legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "roc_curves_top_models.png", dpi=200, bbox_inches="tight")
plt.show()
display(pd.DataFrame(roc_rows).sort_values("auc", ascending=False))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
pr_rows = []
for model_name in top_curve_models:
    part = all_model_predictions[all_model_predictions["model"] == model_name]
    y_true = part["home_win"].astype(int).to_numpy()
    y_score = part["pred_home_win_probability"].astype(float).to_numpy()
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    pr_rows.append({"model": model_name, "average_precision": ap})
    ax.plot(recall, precision, linewidth=1.8, label=f"{model_name[:34]} ({ap:.3f})")

base_rate = all_model_predictions.drop_duplicates(["match_date", "home_team", "away_team"])["home_win"].mean()
ax.axhline(base_rate, linestyle="--", color="black", linewidth=1, label=f"Base rate ({base_rate:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves for Top Models")
ax.legend(fontsize=7, loc="lower left")
plt.tight_layout()
plt.savefig(FIG_DIR / "precision_recall_curves_top_models.png", dpi=200, bbox_inches="tight")
plt.show()
display(pd.DataFrame(pr_rows).sort_values("average_precision", ascending=False))


In [ ]:
confusion_rows = []
for model_name, part in all_model_predictions.groupby("model"):
    y_true = part["home_win"].astype(int).to_numpy()
    y_pred = part["pred_home_win"].astype(int).to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    confusion_rows.append({
        "model": model_name,
        "family": part["family"].iloc[0],
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "accuracy_percent": (tp + tn) / (tp + tn + fp + fn) * 100,
        "specificity_percent": tn / (tn + fp) * 100 if (tn + fp) else np.nan,
        "recall_percent": tp / (tp + fn) * 100 if (tp + fn) else np.nan,
    })
confusion_summary = pd.DataFrame(confusion_rows).sort_values("accuracy_percent", ascending=False)
display(confusion_summary.head(30))
confusion_summary.to_csv(REPORT_DIR / "all_model_confusion_matrix_summary.csv", index=False)

top_cm = confusion_summary.head(16)
fig, axes = plt.subplots(4, 4, figsize=(13, 12))
axes = axes.ravel()
for ax, (_, row) in zip(axes, top_cm.iterrows()):
    part = all_model_predictions[all_model_predictions["model"] == row["model"]]
    cm = confusion_matrix(part["home_win"], part["pred_home_win"], labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{row['model'][:28]}\nAcc {row['accuracy_percent']:.2f}%", fontsize=8)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Loss", "Win"], fontsize=7)
    ax.set_yticklabels(["Loss", "Win"], fontsize=7)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9, weight="bold")
for ax in axes[len(top_cm):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrices_top_models_grid.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
roc_each_dir = FIG_DIR / "roc_curves_each_model"
cm_each_dir = FIG_DIR / "confusion_matrices_each_model"
roc_each_dir.mkdir(parents=True, exist_ok=True)
cm_each_dir.mkdir(parents=True, exist_ok=True)

for model_name, part in all_model_predictions.groupby("model"):
    y_true = part["home_win"].astype(int).to_numpy()
    y_score = part["pred_home_win_probability"].astype(float).to_numpy()
    y_pred = part["pred_home_win"].astype(int).to_numpy()
    safe = safe_filename(model_name)

    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, color="#4C78A8", label=f"AUC={roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(model_name)
    ax.legend()
    plt.tight_layout()
    plt.savefig(roc_each_dir / f"{safe}_roc.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 3.5))
    ax.imshow(cm, cmap="Blues")
    ax.set_title(model_name, fontsize=8)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Home loss", "Home win"], fontsize=7)
    ax.set_yticklabels(["Home loss", "Home win"], fontsize=7)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", weight="bold")
    plt.tight_layout()
    plt.savefig(cm_each_dir / f"{safe}_confusion.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

print(f"Saved individual ROC curves to: {roc_each_dir}")
print(f"Saved individual confusion matrices to: {cm_each_dir}")


## Ablation Study

The ablation study is important for publication because it shows which part of the system contributes to performance.

The final saved ablation compares:

1. Market-only features.
2. Market plus odds-lag features.
3. Market plus external historical-rating features.

This supports the argument that the model improves because of meaningful cricket-history transfer, not only because it copies bookmaker odds.


In [ ]:
ablation_view = ablation[[
    "ablation", "feature_count", "validation_accuracy", "test_accuracy_val_threshold", "test_auc", "test_brier"
]].copy()
for c in ["validation_accuracy", "test_accuracy_val_threshold", "test_auc"]:
    ablation_view[c + "_percent"] = ablation_view[c] * 100
ablation_view["test_brier"] = ablation_view["test_brier"].round(4)
display(ablation_view)

fig, ax = plt.subplots(figsize=(8, 4.5))
labels = ablation["ablation"].str.replace("_", " ").str.title()
values = ablation["test_accuracy_val_threshold"] * 100
bars = ax.bar(labels, values, color=["#B279A2", "#F58518", "#54A24B"])
ax.set_ylabel("Chronological test accuracy (%)")
ax.set_title("Ablation Study: Feature Source Contribution")
ax.set_ylim(0, 100)
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 1, f"{value:.2f}%", ha="center", va="bottom")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "ablation_test_accuracy.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
top_auc = accuracy.sort_values("test_auc_percent", ascending=False).head(12).copy()
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_auc["model"], top_auc["test_auc_percent"], color="#72B7B2")
ax.invert_yaxis()
ax.set_xlabel("Test AUC (%)")
ax.set_title("Top Models by Chronological Test AUC")
for i, v in enumerate(top_auc["test_auc_percent"]):
    ax.text(v + 0.15, i, f"{v:.2f}%", va="center")
ax.set_xlim(max(0, top_auc["test_auc_percent"].min() - 5), min(100, top_auc["test_auc_percent"].max() + 5))
plt.tight_layout()
plt.savefig(FIG_DIR / "top_models_auc.png", dpi=200, bbox_inches="tight")
plt.show()


## Prediction Behavior Analysis

The next cells analyze the final model's prediction probabilities, confidence, calibration, and errors. This helps in the discussion section because it shows whether the model is only correct on obvious matches or whether it handles uncertain matches reasonably.


In [ ]:
print(f"Prediction rows: {len(predictions)}")
print(f"Observed test accuracy from prediction file: {predictions['correct'].mean() * 100:.4f}%")
display(predictions.head(10))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(predictions.loc[predictions["correct"] == 1, "pred_home_win_probability"], bins=20, alpha=0.65, label="Correct", color="#54A24B")
ax.hist(predictions.loc[predictions["correct"] == 0, "pred_home_win_probability"], bins=20, alpha=0.65, label="Incorrect", color="#E45756")
ax.axvline(summary["best_validation_threshold"], color="black", linestyle="--", linewidth=1.5, label="Validation threshold")
ax.set_xlabel("Predicted home-win probability")
ax.set_ylabel("Match count")
ax.set_title("Prediction Probability Distribution")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "prediction_probability_distribution.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
predictions["confidence_bin"] = pd.qcut(predictions["confidence"], q=5, duplicates="drop")
confidence_table = predictions.groupby("confidence_bin", observed=True).agg(
    matches=("correct", "size"),
    accuracy=("correct", "mean"),
    mean_confidence=("confidence", "mean"),
).reset_index()
confidence_table["accuracy_percent"] = confidence_table["accuracy"] * 100
confidence_table["mean_confidence_percent"] = confidence_table["mean_confidence"] * 100
display(confidence_table)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(len(confidence_table)), confidence_table["accuracy_percent"], marker="o", label="Observed accuracy")
ax.plot(range(len(confidence_table)), confidence_table["mean_confidence_percent"], marker="s", label="Mean confidence")
ax.set_xticks(range(len(confidence_table)))
ax.set_xticklabels([str(x) for x in confidence_table["confidence_bin"]], rotation=20, ha="right")
ax.set_ylabel("Percent")
ax.set_title("Confidence vs Correctness")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "confidence_vs_correctness.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
bins = np.linspace(0, 1, 11)
predictions["probability_bin"] = pd.cut(predictions["pred_home_win_probability"], bins=bins, include_lowest=True)
calibration = predictions.groupby("probability_bin", observed=True).agg(
    matches=("home_win", "size"),
    mean_predicted_probability=("pred_home_win_probability", "mean"),
    observed_home_win_rate=("home_win", "mean"),
).reset_index()
display(calibration)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1, label="Perfect calibration")
ax.scatter(calibration["mean_predicted_probability"], calibration["observed_home_win_rate"], s=calibration["matches"] * 8, color="#4C78A8", alpha=0.75)
ax.set_xlabel("Mean predicted home-win probability")
ax.set_ylabel("Observed home-win rate")
ax.set_title("Calibration Plot")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "calibration_plot.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
error_cases = predictions.loc[predictions["correct"] == 0].copy()
error_cases = error_cases.sort_values("confidence", ascending=False)
cols = [c for c in [
    "match_date", "home_team", "away_team", "home_win", "predicted_winner",
    "pred_home_win_probability", "confidence", "market_home_prob_avg", "market_favorite_prob",
    "external_history_matched"
] if c in error_cases.columns]
display(error_cases[cols].head(20))
print(f"Incorrect predictions: {len(error_cases)} out of {len(predictions)}")


## Generalization Safeguard And Smaller Variant

The 620-tree single-seed market-anchored residual XGBoost remains the locked primary model because it produced the strongest chronological test accuracy in the completed experiment. A three-seed ensemble and a 438-tree validation-loss-selected model are retained as robustness comparisons, not as replacements for the official result.


In [ ]:
GENERALIZATION_FILE = OUTPUT_DIR / "final_model_generalization_comparison.csv"
generalization_comparison = pd.read_csv(GENERALIZATION_FILE)
display(generalization_comparison)

comparison_plot = generalization_comparison.copy()
comparison_plot["label"] = comparison_plot["role"].map({
    "single_seed_primary": "Primary model (620 trees)",
    "seed_ensemble_comparison": "Three-seed comparison",
    "validation_loss_selected_smaller_variant": "Smaller variant (438 trees)",
}).fillna(comparison_plot["role"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(
    comparison_plot["label"],
    comparison_plot["test_accuracy_val_threshold"] * 100,
    color=["#54A24B", "#4C78A8", "#F58518"],
)
axes[0].set_ylabel("Chronological test accuracy (%)")
axes[0].set_title("Accuracy: Generalization Comparison")
axes[0].tick_params(axis="x", rotation=12)

axes[1].bar(
    comparison_plot["label"],
    comparison_plot["test_auc"] * 100,
    color=["#54A24B", "#4C78A8", "#F58518"],
)
axes[1].set_ylabel("Test AUC (%)")
axes[1].set_title("AUC: Generalization Comparison")
axes[1].tick_params(axis="x", rotation=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "generalization_variant_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

primary_row = comparison_plot[
    comparison_plot["role"] == "single_seed_primary"
].iloc[0]
ensemble_row = comparison_plot[
    comparison_plot["role"] == "seed_ensemble_comparison"
].iloc[0]
smaller_row = comparison_plot[
    comparison_plot["role"] == "validation_loss_selected_smaller_variant"
].iloc[0]
print(
    "Three-seed comparison accuracy change vs primary:",
    round(
        (
            ensemble_row["test_accuracy_val_threshold"]
            - primary_row["test_accuracy_val_threshold"]
        )
        * 100,
        4,
    ),
    "percentage points",
)
print(
    "Three-seed comparison AUC change vs primary:",
    round((ensemble_row["test_auc"] - primary_row["test_auc"]) * 100, 4),
    "percentage points",
)


## Robustness And Overfitting Diagnostics

A publishable report should not only show the best score. It should show whether the best model is stable across validation and test, whether the validation-to-test gap is large, and whether very high validation scores fail to generalize.


In [ ]:
diagnostic = accuracy.copy()
diagnostic["absolute_gap_percent"] = diagnostic["validation_test_accuracy_gap_percent"].abs()
diagnostic = diagnostic.sort_values(["test_accuracy_val_threshold_percent", "absolute_gap_percent"], ascending=[False, True])
display(diagnostic[[
    "model", "family", "validation_accuracy_percent",
    "test_accuracy_val_threshold_percent", "validation_test_accuracy_gap_percent",
    "absolute_gap_percent", "test_auc_percent", "test_brier_percent"
]].head(25))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(diagnostic["validation_test_accuracy_gap_percent"], bins=20, color="#F58518", alpha=0.85)
axes[0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Validation-Test Accuracy Gap Distribution")
axes[0].set_xlabel("Validation - test accuracy gap (percentage points)")
axes[0].set_ylabel("Models")

axes[1].scatter(diagnostic["test_auc_percent"], diagnostic["test_accuracy_val_threshold_percent"], color="#4C78A8", alpha=0.8)
axes[1].set_title("AUC vs Test Accuracy")
axes[1].set_xlabel("Test AUC (%)")
axes[1].set_ylabel("Test accuracy (%)")

plt.tight_layout()
plt.savefig(FIG_DIR / "robustness_gap_and_auc_accuracy.png", dpi=200, bbox_inches="tight")
plt.show()


## Main Results Draft

The locked primary model achieved 85.3556% accuracy on the chronological held-out test set. The test period contains the newest matches, from 2022-07-22 to 2026-04-23, which makes the result closer to real-world forecasting than a random split. The model also achieved 90.1687% test AUC, a Brier score of 0.1234, and a validation-to-test accuracy gap of -1.0625 percentage points.

The confusion matrix contains 70 correct class-0 predictions, 134 correct class-1 predictions, 24 false-positive predictions, and 11 false-negative predictions. Validation accuracy was 84.2932%. The learning-curve section separately reports training behavior without using the chronological test set for tuning.

The model, threshold, and architecture were selected using training and validation information. The held-out test result is reported once as the principal estimate of future-period performance.


In [ ]:
report_paragraph = (
    f"The proposed {summary['best_family']} model ({summary['best_model']}) achieved "
    f"{summary['best_test_accuracy_percent']:.4f}% chronological test accuracy and "
    f"{summary['best_test_auc'] * 100:.4f}% AUC. The chronological split trained on "
    f"matches from {summary['train_date_range']['start']} to {summary['train_date_range']['end']}, "
    f"used validation matches from {summary['validation_date_range']['start']} to "
    f"{summary['validation_date_range']['end']}, and tested on matches from "
    f"{summary['test_date_range']['start']} to {summary['test_date_range']['end']}. "
    "The leakage audit passed, with no final result, winner label, or score-like feature "
    "used in the model feature sets. "
    f"The validation-test accuracy gap was {summary['best_validation_test_accuracy_gap_percent']:.4f} "
    "percentage points, suggesting that the selected model did not show severe overfitting "
    "under future-period evaluation."
)
print(report_paragraph)


## Why This Is Different From Normal XGBoost

A normal XGBoost workflow usually gives the algorithm a table of features and asks it to learn the full win/loss mapping directly. The proposed approach changes the architecture in several ways.

1. **Market prior as base margin:** bookmaker probability is used as the starting point. XGBoost learns corrections to the market, not the whole match outcome from zero.
2. **Residual learning:** the cricket-history features explain where the market prior may be too high or too low.
3. **Dual view modeling:** home-team and favorite-team views are both used, making the representation more suitable for cricket than a fixed home/away framing only.
4. **Independent temporal ratings:** external history is converted into pre-match Elo/Glicko/form features, then compared with market probability.
5. **Chronological validation:** the system is evaluated on future matches, which is more realistic and more defensible.

This makes the research XGBoost-centered, but not a simple baseline XGBoost paper.


## Defense-Ready Novelty Points

Use these points when explaining why the work has novelty.

1. The model combines betting-market probability with independent cricket-history ratings instead of relying on only one data source.
2. XGBoost is architecturally modified through market anchoring, where odds probability becomes the base margin.
3. The model learns residual corrections to bookmaker expectations using cricket-specific temporal features.
4. The favorite-side representation handles the favorite/underdog nature of betting markets and cricket outcomes.
5. Train-only encodings and chronological testing reduce leakage and make the result more realistic.
6. The ablation study proves that external history and architecture add value beyond market-only prediction.
7. The leakage audit makes the methodology more defensible than models that silently use future information.


## Limitations And Threats To Validity

The following limitations should be honestly reported.

1. The dataset is focused on ODI cricket, so results may not transfer directly to T20, Test cricket, or domestic leagues.
2. The model depends on market odds. If odds are unavailable, the method needs a non-market variant.
3. Toss and batting-first features must be described according to prediction timing. If the claim is strictly pre-toss prediction, remove those features and rerun the experiment.
4. The chronological result is lower than some random-split experiments, but it is more realistic and more publishable.
5. External validation on newer future matches would make the paper stronger.
6. Bookmaker odds already contain public and private information; the paper should frame the contribution as market correction and hybrid intelligence, not as replacing the market completely.


## Suggested Paper Structure

1. **Title:** Market-Anchored Residual XGBoost with Temporal Cricket Ratings for ODI Match Outcome Prediction.
2. **Abstract:** problem, dataset, model novelty, chronological evaluation, best accuracy/AUC.
3. **Introduction:** why cricket prediction is difficult, why leakage matters, why market odds alone is not enough.
4. **Related Work:** sports prediction, cricket ML, XGBoost, odds-based forecasting, temporal rating systems.
5. **Dataset:** market odds source, match-history source, usable rows, time range, target variable.
6. **Preprocessing:** cleaning, team normalization, odds-to-probability conversion, temporal reconstruction.
7. **Feature Engineering:** market, rating, form, venue, head-to-head, market-rating disagreement, favorite symmetry.
8. **Methodology:** market-anchored residual XGBoost, base margin, validation tuning, final refit.
9. **Experimental Setup:** chronological train/validation/test split, metrics, leakage audit.
10. **Results:** accuracy table, AUC/Brier/F1, confusion matrix, validation-test gap.
11. **Ablation Study:** market-only vs odds-lag vs external history vs full model.
12. **Discussion:** what the model learns, where it fails, real-world applicability.
13. **Limitations:** dataset scope, odds dependency, post-toss timing, future validation.
14. **Conclusion:** summarize contribution and future work.


## Figure And Table Inventory

Use this inventory to make sure the final report has enough visual and tabular evidence. The code below lists every generated figure and the CSV table created for all-model confusion matrices.


In [ ]:
figure_inventory = pd.DataFrame({
    "figure_file": [str(p.relative_to(REPORT_DIR)) for p in sorted(FIG_DIR.rglob("*.png"))]
})
table_inventory = pd.DataFrame({
    "table_file": [str(p.relative_to(REPORT_DIR)) for p in sorted(REPORT_DIR.glob("*.csv"))]
})
print("Figures:")
display(figure_inventory)
print("Generated report tables:")
display(table_inventory)


## Publication Checklist

Before submitting the research paper, confirm the following.

- The final reported score is the chronological test accuracy, not a test-tuned exploratory score.
- The paper clearly states whether prediction is pre-toss or post-toss/pre-outcome.
- No final scores, match winners, result columns, or future-derived team stats are included as features.
- Hyperparameters, validation threshold, and model selection are decided before test evaluation.
- Accuracy, AUC, Brier score, F1, MCC, and confusion matrix are reported.
- Ablation study is included.
- Figures are exported from this notebook and used in the report.
- Limitations are stated honestly.
- The model is described as market-anchored residual XGBoost, not just ordinary XGBoost.


In [ ]:
print("Figures saved in:", FIG_DIR)
for path in sorted(FIG_DIR.glob("*.png")):
    print("-", path.name)
